In [10]:
from calibration import load_calibration_sentences
from short_embedding import ShortEmbedding

### Phase 2 — Load Calibration Data

In [11]:
import json

with open("../crosslingual/original/merged_queries_vi.json") as f:
    vi_json = json.load(f)

sentences = list(vi_json.values())[:10_000]
print(f"Loaded {len(sentences)} calibration sentences")
print(sentences[:3])

Loaded 10000 calibration sentences
['Đơn vị dự bị động viên là gì?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên thì phải có trách nhiệm như thế nào?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên có trách nhiệm gì?']


In [14]:
# Extract sentences from JSON values (Vietnamese queries)
sentences = list(vi_json.values())[:10_000]
print(f"Loaded {len(sentences)} calibration sentences")
print(sentences[:3])

Loaded 10000 calibration sentences
['Đơn vị dự bị động viên là gì?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên thì phải có trách nhiệm như thế nào?', 'Quân nhân dự bị được xếp trong đơn vị dự bị động viên có trách nhiệm gì?']


### Load Model

In [12]:
model = ShortEmbedding(
    model_name_or_path="jinaai/jina-embeddings-v5-text-small-retrieval",
    n_prune_layers=2,
)
print(type(model._auto_model))
print(f"Layers: {len(model._get_layers())}")
model._get_layers()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3Model'>
Layers: 28


ModuleList(
  (0-27): 28 x Qwen3DecoderLayer(
    (self_attn): Qwen3Attention(
      (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
      (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
      (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
      (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
    )
    (mlp): Qwen3MLP(
      (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  )
)

### Phase 3 — Compute Block Influence (BI) per Layer

In [ ]:
from tqdm.notebook import tqdm

for start in tqdm(range(0, len(sentences), 32)):
    model.eval_importance(sentences[start:start + 32], batch_size=32, angular=False)

print("Layer importances (cosine-based BI):")
for i, score in enumerate(model.importances):
    print(f"  Layer {i:2d}: {score:.4f}")

In [ ]:
# Optional: angular distance metric (from Gromov et al.)
# Resets importances and recomputes using arccos similarity
model.importances = [0.0 for _ in model._get_layers()]
model.eval_importance(sentences, batch_size=32, angular=True)
print("Layer importances (angular BI):")
for i, score in enumerate(model.importances):
    print(f"  Layer {i:2d}: {score:.4f}")

### Phase 4 — Remove Least Important Layers

### FFN Neuron Pruning

In [ ]:
from tqdm.notebook import tqdm

# Measure neuron activation importance across calibration sentences
neuron_scores = model.eval_ffn_importance(sentences, batch_size=32)

for i, scores in enumerate(neuron_scores):
    print(f"Layer {i:2d}: {len(scores)} neurons, mean={scores.mean():.4f}, min={scores.min():.4f}, max={scores.max():.4f}")

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

before = count_params(model._auto_model)

# Prune 20% of FFN neurons per layer — adjust prune_ratio as needed
model.prune_ffn(neuron_scores, prune_ratio=0.2)

after = count_params(model._auto_model)
print(f"Params before: {before:,}")
print(f"Params after:  {after:,}")
print(f"Reduction:     {before - after:,} ({(before - after) / before * 100:.1f}%)")

In [17]:
# Embed a few sentences before pruning for comparison
test_sentences = [
    "The cat sat on the mat.",
    "A quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence.",
]
before_embs = model.get_embeddings(test_sentences)
print(f"Embeddings before pruning: {before_embs.shape}")

Embeddings before pruning: (3, 768)


In [18]:
removed = model.remove_layers()
print(f"Removed layers: {removed}")
print(f"Remaining layers: {len(model._get_layers())}")
model._get_layers()

Removed layers: [21, 19, 20]
Remaining layers: 21


ModuleList(
  (0-20): 21 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=768, out_features=768, bias=False)
      (k_proj): Linear(in_features=768, out_features=256, bias=False)
      (v_proj): Linear(in_features=768, out_features=256, bias=False)
      (o_proj): Linear(in_features=768, out_features=768, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=768, out_features=1152, bias=False)
      (up_proj): Linear(in_features=768, out_features=1152, bias=False)
      (down_proj): Linear(in_features=1152, out_features=768, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((768,), eps=1e-06)
  )
)

In [19]:
import torch

after_embs = model.get_embeddings(test_sentences)
print(f"Embeddings after pruning: {after_embs.shape}")

# Cosine similarity between before/after embeddings to measure drift
before_t = torch.tensor(before_embs)
after_t = torch.tensor(after_embs)
cosine_sim = torch.nn.functional.cosine_similarity(before_t, after_t)
for i, sim in enumerate(cosine_sim):
    print(f"  Sentence {i}: cosine similarity before vs after = {sim:.4f}")

Embeddings after pruning: (3, 768)
  Sentence 0: cosine similarity before vs after = 0.7059
  Sentence 1: cosine similarity before vs after = 0.7449
  Sentence 2: cosine similarity before vs after = 0.7379


### Explicit Layer Removal (optional)
Pass specific layer indices instead of relying on importance scores.

In [ ]:
# Example: remove specific layers by index
# model.remove_layers(layers_to_remove=[10, 11, 12])

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

save_path = "../crosslingual_pruned"

model.st_model[0].auto_model = model._auto_model
model.st_model.save(save_path)
model._tokenizer.save_pretrained(save_path)
model._auto_model.config.save_pretrained(save_path)

snapshot_path = snapshot_download(model.st_model[0].auto_model.config._name_or_path)
for fname in os.listdir(snapshot_path):
    if fname.endswith(".py"):
        shutil.copy2(os.path.join(snapshot_path, fname), os.path.join(save_path, fname))

print(f"Pruned model saved to {save_path}")